In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

version = 5
round = 39

In [7]:
# Set the total number of instance files to process.
num_instances = 33 

# --- Data Aggregation ---
# This list will store the calculated metrics for each instance.
aggregated_metrics = []
time_series_data = {}  # Store full time series for deeper analysis

# Loop through each instance file to calculate and aggregate metrics.
for i in range(1, num_instances + 1):
    # Construct the file path for the current instance.
    file_path = f'../raw_data/v{version}/round_{round}/instance_{i}.csv'
    
    try:
        # Read the instance data from the CSV file.
        df = pd.read_csv(file_path)
        
        # Store full dataframe for later analysis
        time_series_data[i] = df.copy()

        # --- Core Metric Calculations ---

        # Mid Price and Spread
        df["mid_price"] = (df["best_bid"] + df["best_ask"]) / 2
        df["spread"] = df["best_ask"] - df["best_bid"]
        df['price_return'] = df['mid_price'].pct_change(fill_method=None)
        
        # --- Market Making Specific Metrics ---
        
        # 1. Basic Statistics
        final_pnl = df['total_pnl'].iloc[-1]
        max_pnl = df['total_pnl'].max()
        min_pnl = df['total_pnl'].min()
        average_spread = df['spread'].mean()
        spread_std = df['spread'].std()
        
        # 2. Volatility Metrics
        volatility = df['price_return'].std()
        realized_vol = np.sqrt(252) * volatility  # Annualized assuming 252 trading periods
        
        # 3. Position Management Metrics
        avg_position = df['position'].abs().mean()
        max_position = df['position'].abs().max()
        position_utilization = avg_position / 200  # As % of position limit
        position_flips = ((df['position'].shift(1) * df['position']) < 0).sum()  # Times position changed sign
  
        # Store all metrics
        instance_metrics = {
            'instance': i,
            'final_pnl': final_pnl,
            'max_pnl': max_pnl,
            'min_pnl': min_pnl,
            'volatility': volatility,
            'realized_vol': realized_vol,
            'average_spread': average_spread,
            'spread_std': spread_std,
            'avg_position': avg_position,
            'max_position': max_position,
            'position_utilization': position_utilization,
            'position_flips': position_flips,
            'pnl_per_unit_risk': final_pnl / max(max_position, 1)
        }
        aggregated_metrics.append(instance_metrics)

    except FileNotFoundError:
        print(f"File not found: {file_path}. Skipping.")
    except Exception as e:
        print(f"An error occurred while processing {file_path}: {e}")

# Convert to DataFrame
metrics_df = pd.DataFrame(aggregated_metrics)
print(f"Successfully processed {len(metrics_df)} instances")

Successfully processed 33 instances


In [ ]:
pd.

AttributeError: module 'pandas' has no attribute 'head'